<h1 style="text-align: center; color: white; background-color: #1e86cc; border-radius: 30px; padding: 10px;">COSC3068 - Deep Learning</h1>
<h2 style="text-align: center;">Project 1: Letter Classification - Reduced Notebook</h2>
<h3 style="text-align: center;">Author: Nguyen Gia Khang</h3>
<hr />

> I declare that this submission is my own work. I have acknowledged all sources of code, pretrained models, and other materials, and disclosed any use of AI tools in accordance with the course requirements.

## 1. Define and Train

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
MODEL_FILE = "s4034066_project1_model.keras"
CLASS_NAMES = np.array(["a", "b", "c", "d", "e", "f", "g", "h", "k"])
tf.keras.utils.set_random_seed(SEED)

with np.load("student_dataset.npz") as data:
    X = data["images"].astype(np.float32)
    labels = data["labels"].astype(str)

if X.ndim != 3 or X.shape[1:] != (32, 32):
    raise ValueError("Expected images with shape (N, 32, 32).")
if not np.isin(labels, CLASS_NAMES).all():
    raise ValueError("Dataset contains an unexpected label.")

label_to_index = {label: i for i, label in enumerate(CLASS_NAMES)}
y = np.array([label_to_index[label] for label in labels], dtype=np.int32)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05, seed=SEED),
    tf.keras.layers.RandomTranslation(0.08, 0.08, seed=SEED),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32)),
    tf.keras.layers.Reshape((32, 32, 1)),
    data_augmentation,
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )],
    verbose=2,
)
model.save(MODEL_FILE)

def predict_fn(X32x32: np.ndarray) -> np.ndarray:
    X32x32 = np.asarray(X32x32, dtype=np.float32)
    if X32x32.ndim != 3 or X32x32.shape[1:] != (32, 32):
        raise ValueError("Expected images with shape (N, 32, 32).")
    probabilities = model.predict(X32x32, verbose=0)
    return CLASS_NAMES[np.argmax(probabilities, axis=1)]


## 2. Load the Saved Model

In [ ]:
import numpy as np
import tensorflow as tf

MODEL_FILE = "s4034066_project1_model.keras"
CLASS_NAMES = np.array(["a", "b", "c", "d", "e", "f", "g", "h", "k"])
model = tf.keras.models.load_model(MODEL_FILE)

def predict_fn(X32x32: np.ndarray) -> np.ndarray:
    X32x32 = np.asarray(X32x32, dtype=np.float32)
    if X32x32.ndim != 3 or X32x32.shape[1:] != (32, 32):
        raise ValueError("Expected images with shape (N, 32, 32).")
    probabilities = model.predict(X32x32, verbose=0)
    return CLASS_NAMES[np.argmax(probabilities, axis=1)]


## 3. Predict and Evaluate

In [ ]:
import numpy as np

with np.load("instructor_dataset.npz") as data:
    X_test = data["images"].astype(np.float32)
    y_test = data["labels"].astype(str)

predictions = predict_fn(X_test)
accuracy = np.mean(predictions == y_test)
print(f"Classification accuracy: {accuracy:.4f} ({accuracy:.2%})")
